In [1]:
#pip install transformers torch

In [2]:
from transformers import BertTokenizer, BertModel
import torch
import torch.nn.functional as F

# Load pre-trained model tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize input
input_text = "chicken chop"
tokens = tokenizer(input_text, return_tensors='pt') #if want to remove [CLS]&[SEP], add add_special_tokens=False behind 'pt' 

# Display tokenized input
print(tokens)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

{'input_ids': tensor([[  101,  7975, 24494,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1]])}


c:\Users\Jack\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Jack\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [3]:
# Load pre-trained model
model = BertModel.from_pretrained('bert-base-uncased')

# Get the hidden states (last layer output)
with torch.no_grad():
    outputs = model(**tokens)
    last_hidden_state = outputs.last_hidden_state

# The representation of the [CLS] token
cls_representation = last_hidden_state[:, 0, :]

# Display the representation of the [CLS] token
print(cls_representation.shape)  # Should be (1, 768) for BERT-base

torch.Size([1, 768])


In [4]:
# Example menu items
menu_items = ["mushroom chicken chop", "beef chop", "chicken spaghetti", "chicken wings"]
menu_item_tokens = tokenizer(menu_items, return_tensors='pt', padding=True, truncation=True)

# Get the hidden states for menu items
with torch.no_grad():
    menu_outputs = model(**menu_item_tokens)
    menu_last_hidden_state = menu_outputs.last_hidden_state

# The representation of the [CLS] token for each menu item
menu_cls_representations = menu_last_hidden_state[:, 0, :]

# Calculate cosine similarity
similarities = F.cosine_similarity(cls_representation, menu_cls_representations)

# Display similarities
print(similarities)



tensor([0.9659, 0.9684, 0.8805, 0.8400])


In [5]:
# Rank menu items based on similarity
_, top_indices = torch.topk(similarities, k=2)

# Display top similar menu items
top_menu_items = [menu_items[idx] for idx in top_indices]
print("Top similar menu items:", top_menu_items)

Top similar menu items: ['beef chop', 'mushroom chicken chop']
